# Single Pattern Kikuchi Band Width Debug Workflow

This tutorial validates one EBSD pattern before running a whole scan. In debug mode the detector opens an interactive diagnostic figure with pattern-space geometry, the rotated extraction frame, the sampled band strip, and the normalized intensity profile used for edge picking.


## Inputs

You need three files:

- one pattern image (`.png`, `.bmp`, `.tif`, etc.) or a tiny `.npy`;
- one JSON annotation file containing the candidate `central_line` entries;
- one detector YAML with phase and detector settings.

For CTF workflows, generate annotations automatically with the scan automator first, or provide a trusted line annotation JSON while validating detector settings.


In [ ]:
from pathlib import Path

repo = Path.cwd()
config = repo / "bandDetectorOptionsMagnetiteAccuracyTesting.yml"
pattern = repo / "testData" / "Med_Mn_10k_4x4_00995.png"
annotations = repo / "testData" / "Med_Mn_10k_4x4_00995.json"
output_dir = repo / "outputs" / "single_pattern_debug"
output_dir.mkdir(parents=True, exist_ok=True)

print("Config exists:", config.exists())
print("Pattern exists:", pattern.exists())
print("Annotations exist:", annotations.exists())


## Run The Detector

The command below runs one image as a 1 x 1 scan. `--debug` enables interactive diagnostic plots and sets the detector to show band detection figures. Close each plot window to continue.


In [ ]:
import subprocess, sys

cmd = [
    sys.executable, "-m", "kikuchiBandAnalyzer.band_width.detector_cli",
    "--source", str(pattern),
    "--annotations", str(annotations),
    "--config", str(config),
    "--tile-from-single",
    "--tile-rows", "1",
    "--tile-cols", "1",
    "--raw-output", str(output_dir / "bandOutputData.csv"),
    "--filtered-output", str(output_dir / "filtered_band_data.csv"),
    "--json-output", str(output_dir / "bandOutputData.json"),
    "--debug",
]
print(" ".join(cmd))
# Uncomment to execute interactively:
# subprocess.run(cmd, check=True)


## Inspect Outputs

The raw CSV contains all candidate bands. The filtered CSV keeps the best valid band per pixel. The JSON keeps richer per-band metadata such as `band_profile`, `central_line`, and edge indices.


In [ ]:
import pandas as pd

filtered = output_dir / "filtered_band_data.csv"
if filtered.exists():
    df = pd.read_csv(filtered)
    display(df.head())
else:
    print("Run the detector command first; filtered output does not exist yet:", filtered)


## Acceptance Criteria

A single-pattern debug run is ready for scan-scale use when:

- the central line overlays the expected Kikuchi band;
- the extracted strip crosses the band symmetrically;
- the normalized intensity profile has a clear central peak and two plausible minima;
- PSNR and band width are stable when `rectWidth` and `smoothing_sigma` are varied slightly;
- exported CSV/JSON values match the plot annotations.
